# Customer Support RAG Chatbot — Medallion Architecture

End-to-end Retrieval-Augmented Generation (RAG) system for **24/7 context-aware customer support**. Built on Databricks medallion architecture (Bronze → Silver → Gold) with AI Search (Vector Search) for semantic retrieval and Foundation Model LLMs for response generation.

**Pipeline Overview:**
- **Bronze** — Raw ingestion of FAQs, resolved tickets, and knowledge base articles
- **Silver** — Structured parsing with typed columns and quality filtering
- **Gold** — Embed-ready content with PRIMARY KEY, CDF, and metadata for filters
- **Vector Index** — Delta Sync index with managed embeddings (databricks-gte-large-en)
- **RAG** — Retrieve → Augment → Generate using databricks-meta-llama-3-3-70b-instruct

In [0]:
# ============================================================
# STEP 0a: INSTALL DEPENDENCIES
# databricks-vectorsearch (DEPRECATED) → databricks-ai-search (CURRENT)
# databricks-sdk is pre-installed on Serverless — do NOT reinstall
# ============================================================
%pip install --upgrade databricks-ai-search
dbutils.library.restartPython()      

In [0]:
# ============================================================
# GLOBAL CONFIGURATION
# All downstream cells reference these variables.
# ============================================================

# Unity Catalog Settings
CATALOG_NAME = "main"          # Must be UC (not hive_metastore) for Vector Search
SCHEMA_NAME  = "customer_support"  # Created automatically if it does not exist

# Medallion Table Names (fully qualified)
BRONZE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_support"
SILVER_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_support"
GOLD_TABLE   = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_support"

# Vector Search / AI Search Settings
VS_ENDPOINT_NAME = "ai-vector-compute"  # One endpoint per workspace (quota limit)
VS_INDEX_NAME    = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_support_vs_index"

# Model Endpoints (Databricks Foundation Models — no deployment needed)
EMBEDDING_MODEL   = "databricks-gte-large-en"                # 1024-dim, 8192-token ctx
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

print("Configuration loaded.")
print(f"  Catalog : {CATALOG_NAME}")
print(f"  Schema  : {SCHEMA_NAME}")
print(f"  Tables  : {BRONZE_TABLE} → {SILVER_TABLE} → {GOLD_TABLE}")
print(f"  Index   : {VS_INDEX_NAME}")
print(f"  Models  : {EMBEDDING_MODEL} / {LLM_ENDPOINT_NAME}")

In [0]:
# ============================================================
# CLIENT INITIALIZATION
#
# WorkspaceClient  (databricks-sdk)       → auth, LLM serving, UC operations
# AISearchClient   (databricks-ai-search) → endpoint + index management
#
# Both auto-detect notebook credentials. No tokens needed inside a notebook.
# ============================================================
from databricks.sdk import WorkspaceClient
from databricks.ai_search.client import AISearchClient  # July 2026: replaces VectorSearchClient

# WorkspaceClient wraps all Databricks REST APIs
w = WorkspaceClient()

# AISearchClient manages AI Search endpoints and indexes    
ai_search = AISearchClient(disable_notice=True)  # suppress development auth notice

current_user = w.current_user.me()
print(f"Authenticated as : {current_user.user_name}")
print(f"Workspace host   : {w.config.host}")
print(f"Clients ready    : WorkspaceClient + AISearchClient")

In [0]:
# ============================================================
# BRONZE LAYER: Raw Ingestion of Customer Support Data
#
# Ingests raw customer support knowledge base content:
#   - FAQ entries (question + answer)
#   - Resolved support tickets (issue + resolution)
#   - Knowledge base articles (title + body + category)
#
# Schema: record_type, question_or_title, answer_or_body, category,
#         product, severity, ticket_id, source_file, ingested_at
# Zero transformations — preserve everything for full lineage.
# ============================================================
from pyspark.sql.functions import col, current_timestamp, lit

# Create UC catalog + schema (idempotent)
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
except Exception:
    pass  # catalog may already exist under a different owner
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

# ---- Synthetic Customer Support Knowledge Base ----
# In production, replace with spark.read from S3/ADLS/DBFS source files.
support_records = [
    # --- FAQ ENTRIES ---
    ("faq", "How do I reset my password?", "To reset your password, click on the 'Forgot Password' link on the login page. Enter your registered email address and we will send you a reset link. Click the link in the email and follow the instructions to set a new password. The link expires in 30 minutes.", "Account", "Auth", "low", None),
    ("faq", "How do I enable two-factor authentication?", "Go to Settings > Security > Two-Factor Authentication. Choose your preferred method: SMS, authenticator app, or email. Enter the verification code sent to your chosen method. Once verified, 2FA will be enabled for all future logins. We recommend using an authenticator app for better security.", "Account", "Auth", "medium", None),
    ("faq", "What payment methods do you accept?", "We accept Visa, Mastercard, American Express, Discover, PayPal, Apple Pay, Google Pay, and bank transfers (ACH). For enterprise plans, we also accept wire transfers and purchase orders. All transactions are encrypted and PCI-DSS compliant.", "Billing", "Payments", "low", None),
    ("faq", "How do I cancel my subscription?", "To cancel your subscription, go to Settings > Billing > Manage Subscription and click 'Cancel Subscription'. Your access continues until the end of the current billing cycle. No cancellation fees apply. You can reactivate anytime within 90 days without losing your data.", "Billing", "Subscriptions", "low", None),
    ("faq", "How do I upgrade my plan?", "Navigate to Settings > Billing > Plans. Select your desired plan and click 'Upgrade'. The upgrade takes effect immediately and you will be billed a prorated amount for the remainder of the current billing cycle. Downgrades take effect at the start of the next billing cycle.", "Billing", "Subscriptions", "low", None),
    ("faq", "How do I export my data?", "Go to Settings > Data Management > Export Data. Choose the format (CSV, JSON, or XML), select the data categories you want to export, and click 'Export'. You will receive an email with a download link when the export is ready. Exports are available for 7 days.", "Data", "Data Management", "low", None),
    ("faq", "What is the API rate limit?", "Free tier: 100 requests/minute. Pro tier: 1,000 requests/minute. Enterprise tier: 10,000 requests/minute. Rate limit headers are included in every response. If you exceed the limit, you will receive a 429 Too Many Requests response with a Retry-After header.", "API", "Rate Limits", "medium", None),
    ("faq", "How do I integrate with Slack?", "Go to Settings > Integrations > Slack. Click 'Add to Slack' and authorize the app. Once connected, you can receive notifications in Slack channels for new tickets, status changes, and mentions. You can configure which notifications to receive in the integration settings.", "Integrations", "Slack", "low", None),
    ("faq", "How do I create a custom report?", "Navigate to Analytics > Reports > New Report. Select your data source, choose metrics and dimensions, apply filters, and set the time range. You can schedule reports to run automatically and be emailed to stakeholders. Reports support bar charts, line charts, pie charts, and tables.", "Analytics", "Reporting", "low", None),
    ("faq", "How do I invite team members?", "Go to Settings > Team > Invite Members. Enter email addresses separated by commas. Select a role for each invitee (Admin, Editor, Viewer). Invitations expire after 7 days. You can resend invitations or revoke pending invites from the same page.", "Account", "Team Management", "low", None),

    # --- RESOLVED SUPPORT TICKETS ---
    ("ticket", "Login page shows 500 internal server error", "Root cause: A recent deployment introduced a bug in the authentication service that caused a null pointer exception when processing SSO tokens. Fix: We rolled back the deployment to the previous stable version (v2.14.3) and deployed a hotfix (v2.14.5) that adds null checks for SSO token processing. No data was lost. Users may need to clear browser cache and cookies before logging in again.", "Platform", "Authentication", "critical", "TKT-001234"),
    ("ticket", "Payment processing fails with error code PAY-4023", "Root cause: The customer's bank was declining the transaction due to insufficient funds, but the error message was not being propagated correctly to the UI. Fix: Updated the payment gateway integration to map bank decline codes to user-friendly messages. The customer was advised to contact their bank or use an alternative payment method. We also added a retry mechanism for transient bank errors.", "Billing", "Payments", "high", "TKT-001235"),
    ("ticket", "API calls timing out intermittently above 500ms", "Root cause: Database connection pool exhaustion during peak hours. The pool was configured for 50 connections but peak load required 200+. Fix: Increased the connection pool size to 300, added connection pooling with idle timeout of 30 seconds, and implemented circuit breakers to fail fast when the pool is exhausted. Also added monitoring alerts for pool utilization above 80%.", "Platform", "API", "high", "TKT-001236"),
    ("ticket", "Data export to CSV produces corrupted file", "Root cause: Special characters in user data (commas, quotes, newlines) were not being properly escaped in CSV export. Fix: Updated the CSV export library to use proper RFC 4180 compliant escaping. Added automated tests for edge cases including unicode characters, embedded quotes, and multiline fields. Deployed in v3.2.1.", "Data", "Data Management", "medium", "TKT-001237"),
    ("ticket", "Slack integration not sending notifications", "Root cause: The Slack OAuth token had expired. The token was valid for 1 year but no renewal alert was configured. Fix: Renewed the OAuth token, set up a renewal reminder 30 days before expiry, and added monitoring for token expiration. Also improved error logging for failed notification deliveries.", "Integrations", "Slack", "medium", "TKT-001238"),
    ("ticket", "Custom report shows stale data from yesterday", "Root cause: The reporting cache was set to refresh every 24 hours but the cache invalidation job was failing silently due to insufficient permissions. Fix: Granted the cache invalidation service account the necessary permissions on the reporting database. Added alerting for cache refresh job failures. Reduced cache TTL to 1 hour for real-time reports.", "Analytics", "Reporting", "medium", "TKT-001239"),
    ("ticket", "Team member cannot access shared dashboard", "Root cause: The user's role was set to Viewer but the dashboard required Editor permissions. Additionally, the dashboard sharing settings were set to Private instead of Team. Fix: Updated the user's role to Editor and changed the dashboard sharing settings to Team. Added a permissions tooltip in the UI to clarify which roles can access which features.", "Account", "Access Control", "low", "TKT-001240"),
    ("ticket", "Webhook deliveries are delayed by 5+ minutes", "Root cause: The webhook delivery queue was backed up due to a spike in event volume during a product launch. The consumer had only 2 workers processing events. Fix: Scaled the webhook consumer to 10 workers, implemented dead letter queues for failed deliveries, and added retry logic with exponential backoff. Also added a webhook delivery dashboard for monitoring.", "API", "Webhooks", "medium", "TKT-001241"),

    # --- KNOWLEDGE BASE ARTICLES ---
    ("article", "Getting Started with SSO Configuration", "This guide walks you through setting up Single Sign-On (SSO) for your organization. Step 1: Navigate to Settings > Security > SSO. Step 2: Choose your identity provider (Okta, Azure AD, Google Workspace, or SAML 2.0 compatible). Step 3: Upload your identity provider metadata XML file or enter the metadata URL. Step 4: Map user attributes (email, name, groups) from your IdP to our platform. Step 5: Test the SSO configuration with a test user. Step 6: Enable SSO for all users. Note: Once SSO is enabled, password-based login is disabled. Ensure all users have accounts in your identity provider before enabling.", "Platform", "Security", "medium", None),
    ("article", "Best Practices for API Security", "Securing your API integration is critical. 1) Use API keys with least-privilege scopes. 2) Rotate API keys every 90 days. 3) Use IP allowlisting for production environments. 4) Implement rate limiting on your end to avoid 429 responses. 5) Always use HTTPS endpoints. 6) Store API keys in a secrets manager, never in code. 7) Enable audit logging for all API operations. 8) Use webhook signature verification to validate incoming webhooks. 9) Monitor API usage with the /metrics endpoint. 10) Set up alerts for unusual API activity patterns.", "API", "Security", "high", None),
    ("article", "Database Backup and Recovery Guide", "This document covers backup and recovery procedures. Automated Backups: Full backups run daily at 2:00 AM UTC. Incremental backups run every 6 hours. Backups are retained for 30 days. Manual Backups: You can trigger a manual backup from Settings > Data > Backup Now. Recovery: To restore from backup, contact support with the backup timestamp. Point-in-time recovery is available up to 7 days. For enterprise customers, cross-region replication is enabled by default with an RPO of 15 minutes. Disaster Recovery: In case of a region outage, traffic is automatically routed to the standby region. RTO is 30 minutes.", "Data", "Backup Recovery", "high", None),
    ("article", "Understanding Billing and Usage Reports", "Your billing dashboard provides detailed insights into your usage. Billing Cycle: Monthly, charged on the 1st of each month. Usage Metrics: API calls, storage used, active users, and data transfer. Cost Breakdown: See per-service costs in Settings > Billing > Cost Breakdown. Budget Alerts: Set spending thresholds to receive email alerts. Invoices: Download PDF invoices from Settings > Billing > Invoices. Tax Documents: Access annual tax summaries and W-9 forms. For billing disputes, create a support ticket with the Billing category and include the disputed invoice number.", "Billing", "Billing Reports", "low", None),
    ("article", "Setting Up Webhook Endpoints", "Webhooks allow you to receive real-time event notifications. Step 1: Go to Settings > API > Webhooks. Step 2: Click 'Add Endpoint' and enter your callback URL. Step 3: Select the events you want to subscribe to (ticket.created, ticket.updated, payment.processed, etc.). Step 4: Configure retry settings (max retries, backoff strategy). Step 5: Add a signing secret for payload verification. Step 6: Test the endpoint with a sample payload. Webhook delivery: Events are delivered via HTTPS POST with a JSON body. The X-Signature header contains an HMAC-SHA256 signature of the payload using your signing secret. Failed deliveries are retried 3 times with exponential backoff.", "API", "Webhooks", "low", None),
    ("article", "Data Privacy and GDPR Compliance", "We are fully GDPR compliant. Data Subject Rights: Access, Rectification, Erasure, Portability. To process a data subject request, go to Settings > Privacy > Data Subject Requests. Data Retention: Account data is retained for the duration of your subscription. After cancellation, data is held for 90 days then permanently deleted. Data Processing Addendum (DPA): Available for download from Settings > Legal > DPA. Sub-processors: A full list of sub-processors is maintained at Settings > Legal > Sub-processors. Data Residency: EU customers can select EU-only data residency. Cross-border transfers use Standard Contractual Clauses (SCCs).", "Data", "Privacy GDPR", "high", None),
    ("article", "Performance Optimization Tips", "To maximize platform performance: 1) Use pagination for large data sets (max 1000 records per page). 2) Implement caching for frequently accessed data. 3) Use bulk API endpoints for batch operations. 4) Avoid N+1 query patterns by using include/exclude parameters. 5) Compress API responses with gzip. 6) Use webhooks instead of polling for real-time updates. 7) Schedule heavy data exports during off-peak hours. 8) Monitor response times with the X-Response-Time header. 9) Use the /health endpoint for uptime monitoring. 10) Consider our Enterprise tier for dedicated resources and SLA guarantees.", "Platform", "Performance", "low", None),
    ("article", "Mobile App Setup and Troubleshooting", "Our mobile app is available on iOS 14+ and Android 8+. Installation: Download from the App Store or Google Play. Login: Use the same credentials as the web app. SSO is supported. Push Notifications: Enable in Settings > Notifications. Common Issues: If push notifications are not working, check your phone's notification settings and ensure the app is not in battery optimization mode. If the app crashes on startup, clear the app cache and data, then reinstall. If you cannot log in, ensure your account is active and you have an internet connection. For offline access, enable 'Offline Mode' in Settings. Data syncs automatically when connectivity is restored.", "Mobile", "App Setup", "low", None),
]

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)

bronze_schema = StructType([
    StructField("record_type",    StringType(), True),   # faq / ticket / article
    StructField("question_or_title", StringType(), True),
    StructField("answer_or_body",  StringType(), True),
    StructField("category",       StringType(), True),   # Account, Billing, API, etc.
    StructField("product",        StringType(), True),   # Auth, Payments, API, etc.
    StructField("severity",       StringType(), True),   # low / medium / high / critical
    StructField("ticket_id",      StringType(), True),   # TKT-xxxxxx for tickets, NULL for others
])

df_bronze = (
    spark.createDataFrame(support_records, schema=bronze_schema)
    .withColumn("source_file", lit("synthetic_customer_support_kb"))
    .withColumn("ingested_at",  current_timestamp())
)

# Idempotent write
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        df_bronze.write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )
    print(f"BRONZE created : {BRONZE_TABLE}")
else:
    print(f"BRONZE exists  : {BRONZE_TABLE} — skipping write (drop to refresh)")

row_count = spark.table(BRONZE_TABLE).count()
print(f"Rows           : {row_count}")
print(f"Record types   : {[r[0] for r in spark.table(BRONZE_TABLE).select('record_type').distinct().collect()]}")
display(spark.table(BRONZE_TABLE).limit(5))

In [0]:
# ============================================================
# SILVER LAYER: Structured Parsing & Quality Filtering
#
# Transforms Bronze data into typed, analytics-ready columns:
#   - Unifies question_or_title / answer_or_body into clear fields
#   - Adds `source_label` (human-readable record type)
#   - Quality gate: drop records with null content
#   - Deduplicates by content hash
# ============================================================
from pyspark.sql.functions import col, sha2, trim, when, length, concat_ws, lit, lower

df_silver = (
    spark.table(BRONZE_TABLE)
    # Standardize text fields — trim whitespace
    .withColumn("title",   trim(col("question_or_title")))
    .withColumn("body",    trim(col("answer_or_body")))
    .withColumn("category", trim(col("category")))
    .withColumn("product",  trim(col("product")))
    .withColumn("severity", lower(trim(col("severity"))))
    # Add a human-readable source label for the RAG context block
    .withColumn("source_label",
        when(col("record_type") == "faq",     "FAQ Entry")
        .when(col("record_type") == "ticket",  "Resolved Ticket")
        .when(col("record_type") == "article", "Knowledge Base Article")
        .otherwise("Unknown")
    )
    # Add a content hash for deduplication
    .withColumn("content_hash", sha2(concat_ws("|", col("title"), col("body")), 256))
    # Quality gate: drop rows with null or empty body
    .filter(col("body").isNotNull() & (length(trim(col("body"))) > 10))
    .filter(col("title").isNotNull() & (length(trim(col("title"))) > 3))
    # Select final silver schema
    .select(
        "content_hash",
        "record_type",
        "source_label",
        "title",
        "body",
        "category",
        "product",
        "severity",
        "ticket_id",
        "source_file",
        "ingested_at"
    )
    .dropDuplicates(["content_hash"])
)

# Idempotent write
if not spark.catalog.tableExists(SILVER_TABLE):
    (
        df_silver.write
        .format("delta")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"SILVER created : {SILVER_TABLE}")
else:
    print(f"SILVER exists  : {SILVER_TABLE} — skipping write")

row_count = spark.table(SILVER_TABLE).count()
print(f"Parsed rows    : {row_count}")
print(f"\nRecord type distribution:")
spark.table(SILVER_TABLE).groupBy("record_type").count().orderBy("count", ascending=False).show()
display(spark.table(SILVER_TABLE).limit(5))

In [0]:
# ============================================================
# GOLD LAYER: Embed-Ready Enrichment
#
# Requirements for Vector Search Delta Sync:
#   1. PRIMARY KEY column  (doc_id — SHA-256 of content)
#   2. CDF enabled         (delta.enableChangeDataFeed = true)
#   3. Rich text column    (`content`) for managed embedding
#   4. Metadata columns    for similarity_search() pre-filters
#
# The `content` column combines title + body into a natural language
# passage optimized for semantic embedding.
# ============================================================
from pyspark.sql.functions import sha2, concat_ws, lit, coalesce, when, col

df_gold = (
    spark.table(SILVER_TABLE)
    # Stable content-based PRIMARY KEY
    .withColumn("doc_id", col("content_hash"))
    # Rich text content column for embedding — natural language format
    .withColumn("content",
        concat_ws("\n\n",
            concat_ws(": ", lit("Title"), col("title")),
            concat_ws(": ", lit("Type"),  col("source_label")),
            concat_ws(": ", lit("Category"), col("category")),
            concat_ws(": ", lit("Product"), col("product")),
            concat_ws("\n", lit("Content:"), col("body"))
        )
    )
    # Severity bucket for similarity search filters
    .withColumn("severity_level",
        when(col("severity") == "critical", "CRITICAL")
        .when(col("severity") == "high",    "HIGH")
        .when(col("severity") == "medium",  "MEDIUM")
        .when(col("severity") == "low",     "LOW")
        .otherwise("INFO")
    )
    .select(
        "doc_id",           # PRIMARY KEY (SHA-256 hash)
        "content",          # Embedding source column
        "record_type",      # Metadata filter: faq / ticket / article
        "source_label",     # Metadata filter: human-readable type
        "title",            # For display in RAG context
        "body",             # For display in RAG context
        "category",         # Metadata filter
        "product",          # Metadata filter
        "severity",         # Metadata filter (raw)
        "severity_level",   # Metadata filter (bucketed)
        "ticket_id",         # For ticket lookups
        "source_file",
        "ingested_at"
    )
    .dropDuplicates(["doc_id"])
)

# Idempotent write
if not spark.catalog.tableExists(GOLD_TABLE):
    (
        df_gold.write
        .format("delta")
        .saveAsTable(GOLD_TABLE)
    )
    print(f"GOLD created   : {GOLD_TABLE}")
else:
    print(f"GOLD exists    : {GOLD_TABLE} — skipping write")

# Enable Change Data Feed — required for Delta Sync index
spark.sql(f"ALTER TABLE {GOLD_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# PRIMARY KEY requires NOT NULL column — enforce nullability first
try:
    spark.sql(f"ALTER TABLE {GOLD_TABLE} ALTER COLUMN doc_id SET NOT NULL")
    print("doc_id set NOT NULL.")
except Exception as exc:
    if "already" in str(exc).lower() or "cannot" in str(exc).lower():
        print("doc_id nullability already set.")
    else:
        raise

# Add PRIMARY KEY constraint
try:
    spark.sql(f"""
        ALTER TABLE {GOLD_TABLE}
        ADD CONSTRAINT gold_support_pk PRIMARY KEY (doc_id)
    """)
    print("PRIMARY KEY constraint added.")
except Exception as exc:
    if "already exists" in str(exc).lower():
        print("PRIMARY KEY constraint already exists.")
    else:
        raise

row_count = spark.table(GOLD_TABLE).count()
props = spark.sql(f"DESCRIBE DETAIL {GOLD_TABLE}").select("properties").collect()[0][0]
print(f"\nGOLD  : {GOLD_TABLE}")
print(f"Rows  : {row_count}")
print(f"CDF   : {props.get('delta.enableChangeDataFeed', 'NOT SET')}")
print(f"\nSample content column (what gets embedded):")
display(spark.table(GOLD_TABLE).select("doc_id", "content", "category", "severity_level").limit(3))

In [0]:
# ============================================================
# AI SEARCH: Get or Create Endpoint + Delta Sync Index
#
# Creates a Delta Sync index with managed embeddings:
#   - Embedding model: databricks-gte-large-en (1024-dim)
#   - Source table: Gold layer with PRIMARY KEY + CDF
#   - Pipeline type: TRIGGERED (manual .sync() calls)
#
# Workspace quota = 1 endpoint. Reuse existing if available.
# ============================================================
import time

# ---- Get or Create Endpoint ----
def get_or_create_vs_endpoint(client, endpoint_name, endpoint_type="STANDARD"):
    """Get existing endpoint or create and poll until ONLINE."""
    try:
        ep    = client.get_endpoint(name=endpoint_name)
        state = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
        print(f"[FOUND] Endpoint '{endpoint_name}' state: {state}")
        return ep
    except Exception as exc:
        if any(x in str(exc).lower() for x in ["does not exist", "not found", "resource_does_not_exist"]):
            print(f"[CREATE] Creating '{endpoint_name}' (type={endpoint_type})...")
            client.create_endpoint(name=endpoint_name, endpoint_type=endpoint_type)
            print("Polling until ONLINE (typically 5-10 min)...")
            for i in range(40):
                ep    = client.get_endpoint(name=endpoint_name)
                state = ep.get("endpoint_status", {}).get("state", "")
                print(f"  [{i+1:02d}/40] {state}")
                if state == "ONLINE":
                    print("Endpoint ONLINE.")
                    return ep
                time.sleep(30)
            raise TimeoutError("Endpoint did not reach ONLINE in 20 min.")
        raise

vs_endpoint = get_or_create_vs_endpoint(ai_search, VS_ENDPOINT_NAME, "STANDARD")
print(f"\nEndpoint : {vs_endpoint.get('name')}")
print(f"Type     : {vs_endpoint.get('endpoint_type')}")
print(f"State    : {vs_endpoint.get('endpoint_status', {}).get('state')}")

# ---- Get or Create Delta Sync Index ----
def get_or_create_delta_sync_index(
    client, endpoint_name, index_name, source_table,
    primary_key, embedding_col, embedding_model,
    pipeline_type="TRIGGERED"
):
    """Get an existing VS index or create a Delta Sync index with managed embeddings."""
    try:
        idx    = client.get_index(index_name=index_name)  # July 2026: no endpoint_name
        status = idx.describe().get("status", {})
        print(f"[FOUND] Index '{index_name}'")
        print(f"  Ready        : {status.get('ready', False)}")
        print(f"  Indexed rows : {status.get('indexed_row_count', 0):,}")
        return idx
    except Exception as exc:
        if any(x in str(exc).lower() for x in ["does not exist", "not found", "resource_does_not_exist"]):
            print(f"[CREATE] Creating index '{index_name}'")
            client.create_delta_sync_index(
                endpoint_name            = endpoint_name,
                index_name               = index_name,
                source_table_name        = source_table,
                pipeline_type            = pipeline_type,
                primary_key              = primary_key,
                embedding_source_column  = embedding_col,
                embedding_model_endpoint_name = embedding_model,
                # All columns needed in similarity_search() results + filters
                columns_to_sync=[
                    "doc_id", "content", "record_type", "source_label",
                    "title", "body", "category", "product",
                    "severity", "severity_level", "ticket_id"
                ]
            )
            print("\nPolling until ready (may take several min for initial snapshot)...")
            for i in range(40):
                idx    = client.get_index(index_name=index_name)
                status = idx.describe().get("status", {})
                ready  = status.get("ready", False)
                rows   = status.get("indexed_row_count", 0)
                msg    = status.get("message", "")
                print(f"  [{i+1:02d}/40] ready={ready} rows={rows:,} | {msg}")
                if ready:
                    print(f"\nIndex ready with {rows:,} rows.")
                    return idx
                time.sleep(30)
            raise TimeoutError("Index did not become ready in 20 min.")
        raise

vs_index = get_or_create_delta_sync_index(
    client          = ai_search,
    endpoint_name   = VS_ENDPOINT_NAME,
    index_name      = VS_INDEX_NAME,
    source_table    = GOLD_TABLE,
    primary_key     = "doc_id",
    embedding_col   = "content",
    embedding_model = EMBEDDING_MODEL,
    pipeline_type   = "TRIGGERED"
)

In [0]:
# ============================================================
# TRIGGER INDEX SYNC
#
# For TRIGGERED pipeline type, call .sync() to push Gold table updates
# into the vector index. Skip for CONTINUOUS pipelines.
# Handle PROVISIONING_INITIAL_SNAPSHOT state gracefully.
# ============================================================
import time

idx    = ai_search.get_index(index_name=VS_INDEX_NAME)
status = idx.describe().get("status", {})
state  = status.get("detailed_state", "")
ready  = status.get("ready", False)
rows   = status.get("indexed_row_count", 0)

if "PROVISIONING" in state:
    # Initial sync is already running automatically
    prov = status.get("provisioning_status", {}).get("initial_pipeline_sync_progress", {})
    pct  = prov.get("sync_progress_completion", 0) * 100
    eta  = prov.get("estimated_completion_time_seconds", 0)
    print(f"Index provisioning ({state}):")
    print(f"  Rows indexed : {rows:,} / {prov.get('total_rows_to_sync', '?'):,}")
    print(f"  Progress     : {pct:.1f}%")
    print(f"  ETA          : {eta/60:.0f} min")
    print("Similarity search works on the indexed portion.")
elif ready:
    print(f"Index READY with {rows:,} rows. Triggering manual sync...")
    idx.sync()
    print("Sync triggered. Wait 1-2 min for incremental updates.")
else:
    print(f"Index state: {state} | Rows: {rows:,} | {status.get('message', '')}")

In [0]:
# ============================================================
# RETRIEVAL: Similarity Search Function
#
# Searches the Gold vector index for customer support content
# semantically similar to the user's query.
#
# Supports:
#   - ANN (pure semantic) and HYBRID (semantic + keyword BM25)
#   - Metadata pre-filters (category, record_type, severity_level)
#
# July 2026 API:
#   - filters accepts dict directly (no json.dumps)
#   - get_index takes only index_name (no endpoint_name)
#   - disable_notice=True suppresses auth notice
# ============================================================

def similarity_search(
    query:       str,
    top_k:       int  = 5,
    filter_dict: dict = None,
    query_type:  str  = "ANN"
) -> list:
    """
    Search the Gold vector index for support content similar to `query`.

    Args:
        query:       Natural language search string (customer question)
        top_k:       Number of results to return (default 5)
        filter_dict: Metadata pre-filters, e.g. {"category": "Billing"}
                     Multiple keys use AND logic.
        query_type:  "ANN" for pure semantic; "HYBRID" adds keyword matching

    Returns:
        List of dicts, each containing support doc fields + similarity score
    """
    idx = ai_search.get_index(index_name=VS_INDEX_NAME)

    kwargs = dict(
        query_text  = query,
        columns     = ["doc_id", "content", "record_type", "source_label",
                       "title", "body", "category", "product",
                       "severity_level", "ticket_id"],
        num_results = top_k,
        query_type  = query_type
    )
    if filter_dict:
        kwargs["filters"] = filter_dict
    kwargs["disable_notice"] = True

    raw       = idx.similarity_search(**kwargs)
    col_names = [c["name"] for c in raw.get("manifest", {}).get("columns", [])]
    hits      = [dict(zip(col_names, row))
                 for row in raw.get("result", {}).get("data_array", [])]
    return hits

# ---- Quick Test ----
print("Testing similarity search...")
test_hits = similarity_search(
    query  = "How do I reset my password?",
    top_k  = 3
)
for i, h in enumerate(test_hits, 1):
    score = float(h.get("score") or 0.0)
    print(f"  [{i}] score={score:.4f} | type={h.get('record_type'):8} | {str(h.get('title'))[:60]}")
print(f"\nTotal hits: {len(test_hits)}")

In [0]:
# ============================================================
# RAG PIPELINE: Retrieve → Augment → Generate
#
# Full end-to-end RAG for customer support chatbot:
#   1. RETRIEVE  — top-k support docs via similarity_search()
#   2. AUGMENT   — format docs as structured context block
#   3. GENERATE  — call LLM with grounded system + user prompt
#
# July 2026: messages must be ChatMessage SDK objects
# ============================================================
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

def rag_query(
    user_question: str,
    top_k:         int  = 5,
    filter_dict:   dict = None,
    query_type:    str  = "ANN",
    conversation_history: list = None
) -> tuple:
    """
    End-to-end RAG for customer support chatbot.

    Args:
        user_question:        Customer's natural language question
        top_k:                Number of support docs to retrieve (default 5)
        filter_dict:          Metadata pre-filters, e.g. {"category": "Billing"}
        query_type:           "ANN" or "HYBRID"
        conversation_history: List of prior (role, content) tuples for multi-turn context

    Returns:
        (answer: str, retrieved_docs: list)
    """
    # --------------------------------------------------------
    # STEP 1: RETRIEVE
    # --------------------------------------------------------
    docs = similarity_search(
        query       = user_question,
        top_k       = top_k,
        filter_dict = filter_dict,
        query_type  = query_type
    )
    if not docs:
        return "I couldn't find any relevant information in our knowledge base. Please try rephrasing your question or contact our support team directly.", []

    # --------------------------------------------------------
    # STEP 2: AUGMENT — build context block
    # --------------------------------------------------------
    context_lines = []
    for i, d in enumerate(docs, 1):
        context_lines.append(
            f"[{i}] Type: {d.get('source_label', '?')} | Category: {d.get('category', '?')} | "
            f"Product: {d.get('product', '?')} | Severity: {d.get('severity_level', 'N/A')}\n"
            f"    Title: {d.get('title', '?')}\n"
            f"    Content: {str(d.get('body', '?'))[:500]}"
        )
    context = "\n\n".join(context_lines)

    system_prompt = (
        "You are an AI customer support agent for a SaaS platform. "
        "Your role is to resolve customer queries 24/7 with accurate, context-aware responses.\n\n"
        "Guidelines:\n"
        "1. Ground every answer in the provided support knowledge base entries.\n"
        "2. If the retrieved context does not contain the answer, say so honestly — do not hallucinate.\n"
        "3. Provide step-by-step instructions when applicable.\n"
    	"4. Reference the source type (FAQ, Resolved Ticket, or Knowledge Base Article) in your answer.\n"
    	"5. If the issue matches a resolved ticket, summarize the root cause and fix.\n"
    	"6. For critical issues, advise the customer to create a support ticket for immediate escalation.\n"
    	"7. Be concise, friendly, and professional. Use bullet points for clarity.\n"
    	"8. If conversation history is provided, maintain context from previous exchanges."
    )

    user_prompt = (
        f"SUPPORT KNOWLEDGE BASE CONTEXT:\n{context}\n\n"
        f"CUSTOMER QUESTION: {user_question}\n\n"
        "Provide a helpful, accurate response based on the knowledge base entries above."
    )

    # --------------------------------------------------------
    # STEP 3: GENERATE — call LLM
    # July 2026: must use ChatMessage SDK objects, not plain dicts
    # --------------------------------------------------------
    messages = [ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt)]

    # Include conversation history for multi-turn context
    if conversation_history:
        for role, content in conversation_history[-4:]:  # last 4 turns
            msg_role = ChatMessageRole.USER if role == "user" else ChatMessageRole.ASSISTANT
            messages.append(ChatMessage(role=msg_role, content=content))

    messages.append(ChatMessage(role=ChatMessageRole.USER, content=user_prompt))

    response = w.serving_endpoints.query(
        name        = LLM_ENDPOINT_NAME,
        messages    = messages,
        max_tokens  = 1024,
        temperature = 0.3    # Slightly higher than log analysis for friendly tone
    )
    answer = response.choices[0].message.content
    return answer, docs

print("RAG pipeline function `rag_query()` defined.")
print("Supports: single-turn and multi-turn conversations.")
print("Filters: category, record_type, severity_level, product.")

In [0]:
# ============================================================
# INTERACTIVE CHATBOT DEMO
#
# Demonstrates the customer support RAG chatbot with realistic
# customer queries across different categories and severities.
# Shows both single-turn and multi-turn conversation flows.
# ============================================================

print("=" * 70)
print("  CUSTOMER SUPPORT RAG CHATBOT — DEMO QUERIES")
print("=" * 70)

demo_queries = [
    {
        "question":   "I forgot my password and can't log in. How do I reset it?",
        "filters":    {"category": "Account"},
        "top_k":      3,
        "query_type": "ANN"
    },
    {
        "question":   "My payment keeps failing with error code PAY-4023. What should I do?",
        "filters":    {"category": "Billing"},
        "top_k":      3,
        "query_type": "HYBRID"
    },
    {
        "question":   "API calls are timing out during peak hours. Is there a known fix?",
        "filters":    {"category": "Platform"},
        "top_k":      3,
        "query_type": "ANN"
    },
    {
        "question":   "How do I set up Single Sign-On for my organization with Okta?",
        "filters":    {"category": "Platform"},
        "top_k":      3,
        "query_type": "ANN"
    },
    {
        "question":   "My webhook deliveries are delayed. What could be causing this?",
        "filters":    {"category": "API"},
        "top_k":      3,
        "query_type": "HYBRID"
    },
    {
        "question":   "I need to export my data for compliance. What formats are supported?",
        "filters":    {"category": "Data"},
        "top_k":      3,
        "query_type": "ANN"
    },
]

for idx, demo in enumerate(demo_queries, 1):
    print(f"\n{'─' * 70}")
    print(f"CUSTOMER QUERY {idx}: {demo['question']}")
    print(f"FILTERS: {demo['filters']} | Search: {demo['query_type']}")
    print(f"{'─' * 70}")

    answer, docs = rag_query(
        user_question = demo["question"],
        top_k         = demo["top_k"],
        filter_dict   = demo["filters"],
        query_type    = demo["query_type"]
    )

    print(f"\nRetrieved {len(docs)} support docs:")
    for i, d in enumerate(docs, 1):
        score = float(d.get("score") or 0.0)
        print(f"  [{i}] score={score:.4f} | {d.get('source_label', '?'):25} | {str(d.get('title'))[:50]}")

    print(f"\nCHATBOT RESPONSE:\n{answer}")

# ---- Multi-Turn Conversation Demo ----
print(f"\n\n{'=' * 70}")
print("  MULTI-TURN CONVERSATION DEMO")
print(f"{'=' * 70}")

conversation = []
multi_turn_questions = [
    "I want to secure my account better. What options do I have?",
    "How do I enable two-factor authentication?",
    "Can I also set up SSO for my team?",
]

for i, q in enumerate(multi_turn_questions, 1):
    print(f"\n{'─' * 70}")
    print(f"TURN {i} — CUSTOMER: {q}")
    print(f"{'─' * 70}")

    answer, docs = rag_query(
        user_question        = q,
        top_k                = 3,
        conversation_history = conversation
    )

    # Update conversation history
    conversation.append(("user", q))
    conversation.append(("assistant", answer))

    print(f"\nRetrieved {len(docs)} docs:")
    for j, d in enumerate(docs, 1):
        print(f"  [{j}] {d.get('source_label', '?'):25} | {str(d.get('title'))[:50]}")
    print(f"\nCHATBOT:\n{answer}")

print(f"\n{'=' * 70}")
print("  DEMO COMPLETE")
print(f"{'=' * 70}")